# Task 1

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <style>
        body {
            font-family: Arial, sans-serif;
            line-height: 1.6;
            background-color: #f9f9f9;
            color: #333;
            padding: 20px;
        }
        h1, h2, h3 {
            color: #2c3e50;
        }
        .important {
            background-color: #fffae6;
            border-left: 6px solid #f1c40f;
            padding: 10px;
            margin-bottom: 20px;
        }
        .submission {
            background-color: #e8f4fd;
            border-left: 6px solid #2196f3;
            padding: 10px;
            margin-bottom: 20px;
        }
        ul {
            margin-left: 20px;
        }
        .tips {
            background-color: #e8f5e9;
            border-left: 6px solid #4caf50;
            padding: 10px;
            margin-top: 20px;
        }
    </style>
</head>
<body>

<h1>NER Fine-Tuning Task</h1>

<div class="important">
    <h2>Important Setup:</h2>
    <ul>
        <li>Make sure to use <strong>GPU acceleration</strong> if available in your environment for faster training</li>
        <li>You can use <strong>Kaggle</strong> which offers free GPU access for this task</li>
        <li>Save your work frequently and ensure all files are saved properly</li>
    </ul>
</div>

<div class="submission">
    <h2>Submission Requirements:</h2>
    <ul>
        <li>Send your completed work as files (Jupyter Notebook, Python files, and any additional files)</li>
        <li><strong>Include:</strong> Your notebook, any custom datasets, model files, and documentation</li>
        <li>Ensure all code is runnable and well-documented</li>
    </ul>
</div>

<h2>Your Tasks:</h2>
<p><strong>Fine-tune a pretrained model on a Named Entity Recognition (NER) task.</strong></p>

<h3>Keep in mind:</h3>
<ul>
    <li><strong>Loss function</strong></li>
    <li><strong>Optimizer</strong></li>
    <li><strong>Weight initialization</strong> (if applicable)</li>
    <li><strong>Splitting data</strong> into train, validation, and test sets</li>
    <li><strong>Evaluation metrics:</strong> e.g., F1-score, precision, recall</li>
</ul>

<h3>Allowed Resources:</h3>
<p>You are allowed to use:</p>
<ul>
    <li>AI tools</li>
    <li>YouTube tutorials</li>
    <li>Online documentation, blogs, and forums</li>
</ul>
<p><em>In short: Feel free to use the internet for research and implementation.</em></p>

<h3>Dataset Requirements:</h3>
<ul>
    <li><strong>You need to collect or use an Azerbaijani dataset.</strong></li>
</ul>

<h3>Before Starting, Research These Topics:</h3>
<ul>
    <li><strong>Tokenization</strong></li>
    <li><strong>NER labeling schemes</strong> (such as <strong>BIO encoding</strong>)</li>
    <li><strong>Pretrained transformer models for NER:</strong> e.g., BERT, RoBERTa, etc.</li>
</ul>

<h3>Libraries and Frameworks:</h3>
<ul>
    <li>You are allowed to use existing libraries and frameworks:</li>
    <li>Examples: <strong>Hugging Face Transformers, spaCy</strong>, etc.</li>
</ul>

<div class="tips">
    <h3>Recommended:</h3>
    <ul>
        <li>Document your experiment steps, including <strong>hyperparameter choices</strong>, <strong>training logs</strong>, and <strong>evaluation results</strong>.</li>
        <li>If possible, <strong>visualize your results</strong> (e.g., using matplotlib or seaborn for learning curves, confusion matrices, etc.).</li>
    </ul>
</div>

</body>
</html>

In [ ]:
# For example, here is one library that can be used to complete this task (finding a different one, or building fine-tuning from scratch in pytorch is also an option)
# https://flairnlp.github.io/docs/category/tutorial-2-training-models

The code was run in Google Colab environment.

## Prior Inference

In [ ]:
# Prior to the code, let's run an inference.

# 1. First, upload the ZIP file to Colab (or place in working directory)
from google.colab import files
uploaded = files.upload()  # SELECT -> ner_project.zip

In [ ]:
# 2. Unzip the project
!unzip ner_project.zip  # Creates the 3 directories (dataset, model, tokenizer)

# 3. Install requirements
!pip install transformers
!pip install -U datasets

In [ ]:
# 4. Load all components
from transformers import AutoModelForTokenClassification, AutoTokenizer
from datasets import load_from_disk

model = AutoModelForTokenClassification.from_pretrained("./model")
tokenizer = AutoTokenizer.from_pretrained("./tokenizer")
dataset = load_from_disk("./dataset")

# 5. Verify everything loaded
print("Model loaded:", type(model))
print("Tokenizer vocab size:", len(tokenizer))
print("Dataset samples:", dataset["train"][0])

In [ ]:
# run inference

from transformers import pipeline

ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")
ner_pipeline("Microsoft is based in Redmond")

## Code

In [ ]:
# pip installing required dependencies
# After running the cell, you may need to restart the session, if the code is being run in Colab.

!pip install -U datasets
!pip install transformers tokenizers seqeval -q
!pip install evaluate

In [ ]:
# importing libraries

import numpy as np
from transformers import DataCollatorForTokenClassification
from transformers import BertTokenizerFast
from transformers import AutoModelForTokenClassification

In [ ]:
# Loading data

from datasets import load_dataset, load_from_disk

# Login using e.g. `huggingface-cli login` to access this dataset
try:
  ds = load_from_disk("dataset")
except FileNotFoundError:
  ds = load_dataset("LocalDoc/azerbaijani-ner-dataset")
  ds.save_to_disk("dataset")

In [ ]:
# Our data does not have train/val/test split in the beginning.
# We need to split it before we apply transformations as needed.

from datasets import DatasetDict

# First split train into train + temp (80% train, 20% temp)
train_testvalid = ds['train'].train_test_split(test_size=0.2, seed=42)

# Then split temp into validation and test (50% each of the 20%)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

# Combine into final DatasetDict
ds = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],  # Note: 'train' here refers to the first split of test_valid
    'test': test_valid['test']
})

print(ds)

In [ ]:
print(ds['train'].features)
# Should show something like: {'ner_tags': Value('string')}

In [ ]:
# preprocessing data
# we don't need index, because they do not contain any valuable information

ds = ds.select_columns(['tokens', 'ner_tags'])

ds

In [ ]:
# Description of Named Entity tags

chars = '''
0: O: Outside any named entity
1: PERSON: Names of individuals
2: LOCATION: Geographical locations, both man-made and natural
3: ORGANISATION: Names of companies, institutions
4: DATE: Dates or periods
5: TIME: Times of the day
6: MONEY: Monetary values
7: PERCENTAGE: Percentage values
8: FACILITY: Buildings, airports, etc.
9: PRODUCT: Products and goods
10: EVENT: Events and occurrences
11: ART: Artworks, titles of books, songs
12: LAW: Legal documents
13: LANGUAGE: Languages
14: GPE: Countries, cities, states
15: NORP: Nationalities or religious or political groups
16: ORDINAL: Ordinal numbers
17: CARDINAL: Cardinal numbers
18: DISEASE: Diseases and medical conditions
19: CONTACT: Contact information, e.g., phone numbers, emails
20: ADAGE: Proverbs, sayings
21: QUANTITY: Measurements and quantities
22: MISCELLANEOUS: Miscellaneous entities
23: POSITION: Professional or social positions
24: PROJECT: Names of projects or programs
'''

In [ ]:
# Grabbing only the needed information

arbitrary_list = []
for i, char in enumerate(chars.split('\n')):
  arbitrary_list.append(char.split(':'))

arbitrary_list[1:-1]

In [ ]:
ner_tags_id_to_char = {}
ner_tags_char_description = {}
for lst in arbitrary_list[1:-1]:
  ner_tags_id_to_char[lst[0]] = lst[1].strip()
  ner_tags_char_description[lst[1].strip()] = lst[2].strip()

In [ ]:
ner_tags_char_description, ner_tags_id_to_char

In [ ]:
type(ds['train']['ner_tags'][3])

In [ ]:
# Some rows in dataset were type None.
# This datapoints will be a headache, so let's look at how many are there, and get rid of them.

none_indices = [i for i, x in enumerate(ds['train']['ner_tags']) if x is None]
print(f"Found {len(none_indices)} rows with 'ner_tags' = None")

In [ ]:
# Filter out rows where 'ner_tags' is None
ds['train'] = ds['train'].filter(lambda example: example['ner_tags'] is not None and example['tokens'] is not None)

In [ ]:
none_indices = [i for i, x in enumerate(ds['train']['ner_tags']) if x is None]
print(f"Found {len(none_indices)} rows with 'ner_tags' = None")

In [ ]:
# Another important part is to clean the data.
# In Named Entity Recognition, models expect to work with batches, like sequences of lists.
# We have tokens (words) and NER tags. NER tags must be **integers**,
# because they represent the **id's**, and **tokens** should be properly split into words.
# Currently, they look like the following:

ds['train']['tokens']

In [ ]:
ds['train']['ner_tags']

In [ ]:
unique_elements = []
for i in arbitrary_list[1:-1]:
  unique_elements.append(i[1].strip())

unique_elements

In [ ]:
import ast
import numpy as np
from datasets import Dataset, Features, Sequence, ClassLabel, Value

# proper conversion of 'token' features

def convert_string_to_list(example):
    example['tokens'] = ast.literal_eval(str(example['tokens']))
    return example

# proper conversion of 'ner_tags' features
# 1. First convert string lists to actual lists of integers
def convert_tags(example):
    """Safely convert ner_tags to integers, handling None/empty cases"""
    try:
        if example['ner_tags'] is None or str(example['ner_tags']).strip() in ['None', '']:
            return {'ner_tags': []}  # Return empty list for None values

        # Convert string representation to list if needed
        if isinstance(example['ner_tags'], str):
            tag_list = ast.literal_eval(example['ner_tags'])
        else:
            tag_list = example['ner_tags']

        # Ensure all tags are integers
        return {'ner_tags': [int(tag) for tag in tag_list if tag is not None]}

    except Exception as e:
        print(f"Error processing tags: {example['ner_tags']} - {str(e)}")
        return {'ner_tags': []}  # Fallback to empty list

# 2. Define your NER tag classes (customize these to match your labels)
tag_names = unique_elements

# 3. Apply conversion and casting
cleaned_ds = ds.map(convert_string_to_list)
cleaned_ds = cleaned_ds.map(convert_tags, batched=False)
# cleaned_ds = ds.map(convert_tags, batched=False)

cleaned_ds = cleaned_ds.cast_column('ner_tags', Sequence(feature=ClassLabel(names=tag_names)))
cleaned_ds = cleaned_ds.cast_column('tokens', Sequence(feature=Value(dtype='string')))

In [ ]:
# Below is the Correct datatype.

cleaned_ds['train'].features

In [ ]:
# Now, they look like this:

cleaned_ds['train']['tokens']

In [ ]:
# Conversion was succesful!

cleaned_ds['train']['ner_tags']

In [ ]:
# Count of Named Entity tags

ner_tag_count = {}

for example in cleaned_ds['train']:
  for ner_tag in example['ner_tags']:
    ner_tag_count[ner_tag] = ner_tag_count.get(ner_tag, 0) + 1

In [ ]:
ner_tag_count

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Keys will be more understandable, will be used in plotting

ner_tag_count_id_to_char = {ner_tags_id_to_char[str(i)]: j for i, j in zip(ner_tag_count, ner_tag_count.values())}
ner_tag_count_id_to_char

In [ ]:
# Changing the key name from 'O' to 'Undefined'.

dict_copy = ner_tag_count_id_to_char.copy()
dict_copy['Undefined'] = dict_copy.pop('O')
dict_copy

In [ ]:
import pandas as pd

In [ ]:
# Plotting

d = {
    'Named Entity': dict_copy.keys(),
    'Count': dict_copy.values()
}

df = pd.DataFrame(d)
df.sort_values(by='Count', ascending=False).plot.bar(x='Named Entity', y='Count', figsize=(14, 6), rot=45)

plt.show()

# Too much 'Undefined' tags, which is id=0, meaning that we have "Outside any named entity" tags quite a lot.

In [ ]:
# tokenizer is bert-base-uncased, which will be fine-tuned.

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

In [ ]:
cleaned_ds['train'][0]

In [ ]:
# id 101 and 102, are start of sequence and end of sequence tokens(Also seen as [CLS] and [SEP]).
# an example:

example_text = cleaned_ds['train'][0]
tokenized_input = tokenizer(example_text['tokens'], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input['input_ids'])

word_ids = tokenized_input.word_ids()

print(word_ids)

In [ ]:
tokenized_input

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(tokenized_input['input_ids'])
tokens

In [ ]:
len(tokens)

In [ ]:
cleaned_ds['train'][0]['ner_tags']

In [ ]:
def tokenize_and_align_labels(examples):
    """Tokenize and align labels with special handling for None values"""
    # Initialize lists to store results
    batch_input_ids = []
    batch_attention_masks = []
    batch_token_type_ids = []
    batch_labels = []

    for tokens, tags in zip(examples["tokens"], examples["ner_tags"]):
        # Skip None or empty token sequences
        if tokens is None or len(tokens) == 0:
            continue

        # Tokenize (with is_split_into_words=True)
        tokenized = tokenizer(
            tokens,
            truncation=True,
            padding='max_length',
            max_length=128,
            is_split_into_words=True
        )

        # Align labels
        word_ids = tokenized.word_ids()
        aligned_labels = []
        current_word = None

        for word_idx in word_ids:
            # Special tokens get -100
            if word_idx is None:
                aligned_labels.append(-100)
            # New word
            elif word_idx != current_word:
                try:
                    aligned_labels.append(tags[word_idx])
                except IndexError:
                    aligned_labels.append(-100)  # Padding
                current_word = word_idx
            # Same word (subtoken)
            else:
                aligned_labels.append(-100)

        # Store results
        batch_input_ids.append(tokenized["input_ids"])
        batch_attention_masks.append(tokenized["attention_mask"])
        if "token_type_ids" in tokenized:
            batch_token_type_ids.append(tokenized["token_type_ids"])
        batch_labels.append(aligned_labels)

    # Build output dict
    output = {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_masks,
        "labels": batch_labels
    }
    if batch_token_type_ids:
        output["token_type_ids"] = batch_token_type_ids

    return output

In [ ]:
# Apply with filtering of None values first
cleaned_ds = cleaned_ds.filter(lambda x: x["tokens"] is not None and len(x["tokens"]) > 0)

In [ ]:
tokenized_datasets = cleaned_ds.map(tokenize_and_align_labels, batched=True)

In [ ]:
tokenized_datasets['train'][0]

In [ ]:
len(unique_elements)

In [ ]:
# Defining model

model = AutoModelForTokenClassification.from_pretrained('bert-base-uncased', num_labels=25)

In [ ]:
# Defining training arguments for the model

from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    "test-ner",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01
)

In [ ]:
# Initialize a data collator for token classification tasks
# This will dynamically pad batches to the longest sequence in the batch,
# ensuring all inputs have uniform length while preserving labels and attention masks.

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
from evaluate import load

In [ ]:
metric = load("seqeval")

# seqeval -> sequence labeling evaluation

In [ ]:
example = cleaned_ds['train'][0]

In [ ]:
# Extract the list of named entity tag names from the dataset
# The 'feature.names' property contains the human-readable labels for each NER tag

label_list = cleaned_ds['train'].features['ner_tags'].feature.names

In [ ]:
label_list

In [ ]:
# Convert numeric named entity tags to their corresponding label names using label_list

labels = [label_list[i] for i in example['ner_tags']]
labels

In [ ]:
# example of computation

metric.compute(predictions=[labels], references=[labels])

In [ ]:
def compute_metrics(eval_preds):
    """Compute evaluation metrics for token classification (NER) task.

    Args:
        eval_preds: Tuple containing (model_logits, true_labels)

    Returns:
        Dictionary with precision, recall, f1, and accuracy scores
        for the named entity recognition task.
    """
    # Unpack model predictions and true labels
    pred_logits, labels = eval_preds

    # Convert logits to predicted class indices (shape: [batch_size, sequence_length])
    pred_logits = np.argmax(pred_logits, axis=2)

    # Process predictions and labels:
    # 1. Filter out padding tokens (where label = -100)
    # The -100 value comes from PyTorch’s CrossEntropyLoss (used by Hugging Face), which ignores targets with this value.
    # 2. Convert numeric tags to string labels using label_list
    predictions = [
        [label_list[eval_preds] for (eval_preds, l) in zip(prediction, label) if l != -100]
        for (prediction, label) in zip(pred_logits, labels)
    ]

    true_labels = [
        [label_list[l] for (eval_preds, l) in zip(prediction, label) if l != -100]
        for (prediction, label) in zip(pred_logits, labels)
    ]

    # Compute metrics using seqeval's classification_report
    results = metric.compute(predictions=predictions, references=true_labels)

    # Return the main evaluation metrics
    return {
        "precision": results["overall_precision"],  # Precision across all entities
        "recall": results["overall_recall"],        # Recall across all entities
        "f1": results["overall_f1"],               # F1 score across all entities
        "accuracy": results["overall_accuracy"]    # Token-level accuracy
    }

In [ ]:
# Initialize the Hugging Face Trainer for fine-tuning
trainer = Trainer(
    model,  # The pre-trained model to be fine-tuned
    args,   # TrainingArguments containing hyperparameters (lr, batch size, epochs, etc.)

    # Datasets:
    train_dataset=tokenized_datasets['train'],      # Training set (tokenized)
    eval_dataset=tokenized_datasets['validation'],  # Validation set (tokenized)

    # Data processing:
    data_collator=data_collator,  # Collator for dynamic padding/batching
    tokenizer=tokenizer,          # Tokenizer for text processing

    # Evaluation:
    compute_metrics=compute_metrics  # Function to calculate precision/recall/F1
)

# Key Notes:
# - The Trainer handles the entire training loop (forward/backward, logging, etc.)
# - data_collator ensures batches are padded dynamically for efficiency
# - compute_metrics runs during evaluation to track model performance

In [ ]:
# initializing training

trainer.train()

In [ ]:
# Save the fine-tuned model to disk in Hugging Face's standard format
model.save_pretrained("model")

# This creates a directory containing:
# - config.json (model architecture configuration)
# - pytorch_model.bin (model weights)
# - tokenizer files (if saved with tokenizer (which is the cell just below))
# - (optionally) training arguments and special tokens

In [ ]:
# saving the tokenizer

tokenizer.save_pretrained("tokenizer")

In [ ]:
# Create mappings between label IDs and their string representations
# This is necessary for model configuration and output interpretation

# ID to Label mapping (e.g., {0: 'O', 1: 'ORGANIZATION', 2: 'LOCATION', ...})
id2label = {
    str(i): label for i, label in enumerate(label_list)
    # Converts numerical predictions back to human-readable tags
    # Used when decoding model outputs
}

# Label to ID mapping (e.g., {'O': 0, 'ORGANIZATION': 1, 'LOCATION': 2, ...})
label2id = {
    label: str(i) for i, label in enumerate(label_list)
    # Converts string labels to numerical IDs
    # Used when preparing data for model input
}

In [ ]:
id2label

In [ ]:
label2id

In [ ]:
import json

In [ ]:
# Load the model's existing configuration file
config = json.load(open("model/config.json"))

# Update the configuration with our label mappings:
config["id2label"] = id2label  # Add ID to label mapping (e.g., 0 → "O")
config["label2id"] = label2id  # Add label to ID mapping (e.g., "O" → 0)

# Save the updated configuration back to file
json.dump(config, open("model/config.json", "w"))

# Why this matters:
# 1. Ensures the model knows how to interpret its own predictions
# 2. Makes the model self-contained with all needed label information
# 3. Required for proper functioning of pipeline() and model sharing
# 4. Enables correct label display in inference outputs

In [ ]:
# Load the fine-tuned model from disk for inference/continued training

model_fine_tuned = AutoModelForTokenClassification.from_pretrained("model")

In [ ]:
from transformers import pipeline

In [ ]:
# run a simple inference to check

nlp = pipeline("ner", model=model_fine_tuned, tokenizer=tokenizer)

example = "salam corc versene borc"

ner_results = nlp(example)

print(ner_results)

In [ ]:
# Zip all files in the dataset directory

!zip -r ner_project.zip dataset/* model/* tokenizer/*

# Verify contents
# !unzip -l ner_project.zip | head -10  # Show first 10 files

# Task 2

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <style>
        body {
            font-family: Arial, sans-serif;
            line-height: 1.6;
            background-color: #f4f8fb;
            color: #2c3e50;
            padding: 20px;
        }
        h1, h2, h3 {
            color: #34495e;
        }
        .important {
            background-color: #fff3cd;
            border-left: 6px solid #f0ad4e;
            padding: 10px;
            margin-bottom: 20px;
        }
        .bonus {
            background-color: #e0f7fa;
            border-left: 6px solid #00bcd4;
            padding: 10px;
            margin-top: 20px;
        }
        ul {
            margin-left: 20px;
        }
    </style>
</head>
<body>

<h1>Task 2: Voice Activity Detection (VAD) & Speech Emotion Recognition (SER)</h1>

<h2>Your Goal:</h2>
<p>Create a <strong>working example</strong> that demonstrates:</p>
<ul>
    <li><strong>Voice Activity Detection (VAD):</strong> Detecting speech vs. silence in an audio file.</li>
    <li><strong>Speech Emotion Recognition (SER):</strong> Identifying emotional tone in speech (e.g., happy, sad, angry, neutral, etc.).</li>
</ul>

<h3>Audio Input:</h3>
<p>You can use <strong>any audio file</strong> (your voice, downloaded samples, or synthetic speech). The focus is on making a <strong>working example</strong>.</p>

<div class="important">
    <h3>Bonus:</h3>
    <p>If your example works with <strong>Azerbaijani language audio</strong>, you will receive <strong>extra points</strong></p>
</div>

<h3>RRecommended (but not required) Libraries & Tools:</h3>
<ul>
    <li><strong>For VAD:</strong>
        <ul>
            <li>Silero VAD</li>
            <li>WebRTC VAD</li>
            <li>py-webrtcvad</li>
        </ul>
    </li>
    <li><strong>For SER:</strong>
        <ul>
            <li>SpeechBrain</li>
            <li>OpenSMILE (via Python)</li>
            <li>Pretrained Hugging Face SER models</li>
        </ul>
    </li>
</ul>

<h3>Allowed Resources:</h3>
<ul>
    <li>You are <strong>allowed and encouraged</strong> to use any open-source tools, prebuilt models, AI tools, YouTube tutorials, and online guides.</li>
</ul>
</body>
</html>


In [ ]:
# Note:
# The main idea of this task is to understand your interests and see what you can accomplish independently,
# including using AI if you wish. In real projects,
# some research and effort are usually needed to develop working examples,
# so we encourage you to explore and do your best—there’s no pressure to be perfect!

# You can submit your work even if it is not fully completed.

## Voice Activity Detection

In [ ]:
import pandas as pd

In [ ]:
# from google.colab import files
# uploaded = files.upload()   # Upload vad_ser_project.zip

In [ ]:
# !unzip vad_ser_project.zip

In [ ]:
import torch
torch.set_num_threads(1)  # Limit to single CPU thread for stability

from IPython.display import Audio
from pprint import pprint

# Download sample audio file
torch.hub.download_url_to_file('https://models.silero.ai/vad_models/en.wav', 'en_example.wav')

# Load Silero VAD model and utilities
model, utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=True  # Force fresh download
)

# Unpack utility functions
(get_speech_timestamps, _, read_audio, *_) = utils

# Process audio file
sampling_rate = 8000  # Supported rates: 8000 or 16000 Hz

filename = 'orxan_ses.wav'

wav = read_audio(filename, sampling_rate=sampling_rate)  # Read and resample


# Detect speech segments
speech_timestamps = get_speech_timestamps(
    wav,
    model,
    sampling_rate=sampling_rate
)

# Print detected speech segments
pprint(speech_timestamps)  # Pretty-print results


sample_rate = sampling_rate
# from sample intervals to seconds
for segment in speech_timestamps:
    start_sec = segment['start'] / sample_rate
    end_sec = segment['end'] / sample_rate
    print(f"Speech from {start_sec:.2f}s to {end_sec:.2f}s")

In [ ]:
# Azerbaijani voice
Audio(filename)

## Speech Recognition

In [ ]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
import librosa
import librosa.display
from IPython.display import Audio

In [ ]:
def waveplot(data: np.array, sr: int, emotion: str):
    """
    Plot waveform visualization of audio data.

    Parameters:
        data (np.array): Audio time series (1D numpy array)
        sr (int): Sampling rate of audio
        emotion (str): Emotion label for plot title
    """
    plt.figure(figsize=(10,4))  # Set figure size (width, height)
    plt.title(f'Waveform - {emotion}', size=20)  # Add title with emotion label
    librosa.display.waveshow(data, sr=sr)  # Create waveform plot
    plt.show()  # Display the plot

def spectrogram(data: np.array, sr: int, emotion: str):
    """
    Plot spectrogram visualization of audio data.

    Parameters:
        data (np.array): Audio time series (1D numpy array)
        sr (int): Sampling rate of audio
        emotion (str): Emotion label for plot title

    The spectrogram shows frequency content over time, with color representing amplitude (dB).
    """
    # Short-time Fourier Transform (STFT)
    x = librosa.stft(data)  # Convert to frequency domain

    # Convert amplitude to decibel scale for better visualization
    xdb = librosa.amplitude_to_db(abs(x))

    # Create figure
    plt.figure(figsize=(11,4))  # Slightly wider than waveplot
    plt.title(f'Spectrogram - {emotion}', size=20)

    # Display spectrogram
    librosa.display.specshow(
        xdb,
        sr=sr,
        x_axis='time',  # Show time on x-axis
        y_axis='hz'     # Show frequency in Hz on y-axis
    )

    # Add colorbar to show amplitude scale
    plt.colorbar(format='%+2.0f dB')
    plt.show()

In [ ]:
# audio file, sample rate.
a, b = librosa.load('/content/finally.mp3')

waveplot(a, b, 'Aygun Kazimova')
spectrogram(a, b, 'Aygun Kazimova')

In [ ]:
# If df.csv is not available, we will fetch data from kaggle.

def fetch_data_from_kaggle() -> pd.DataFrame:
    # Import the Kaggle Hub library to access datasets
    from time import perf_counter
    t1 = perf_counter()

    import kagglehub


    # Download the Toronto Emotional Speech Set (TESS) dataset from Kaggle
    # - "ejlok1/toronto-emotional-speech-set-tess" is the dataset identifier in format [username]/[dataset-name]
    # - This will download the dataset to a local cache directory
    path = kagglehub.dataset_download("ejlok1/toronto-emotional-speech-set-tess")

    # Print the local path where the dataset was downloaded
    # - This path points to the downloaded dataset files (typically a zip file or directory)
    # - Example output: "/root/.cache/kagglehub/datasets/ejlok1/toronto-emotional-speech-set-tess/1.0.0/"
    print("Path to dataset files:", path)

    paths = []
    labels = []

  # Walk through dataset directory
    for dirname, _, filenames in os.walk(path):
      for filename in filenames:
          # Store full file path
          paths.append(os.path.join(dirname, filename))

          # Extract label from filename (format: prefix_emotion.wav)
          label = filename.split('_')[-1].split('.')[0].lower()
          labels.append(label)

      # Stop after collecting all 2800 files
      if len(paths) == 2800:
          break

    print(f'Loaded {len(paths)} audio files')  # Verify count

    # Dataset verification and summary
    print('Dataset successfully loaded!')
    print(f'Total audio files loaded: {len(paths)}')
    print(f'First 5 labels: {labels[:5]}')  # Show sample of extracted labels
    print(f'Unique emotions detected: {sorted(set(labels))}')  # Show all emotion categories

    df = pd.DataFrame()
    print('Created pd.DataFrame')
    df['speech'] = paths
    df['label'] = labels

    print('Starting data transformation...')
    df['data'] = df['speech'].apply(lambda x: librosa.load(x)[0])  # Audio waveform
    df['sampling_rate'] = df['speech'].apply(lambda x: librosa.load(x)[1])  # Sample rate

    # Remove file path column (they are not needed)
    df = df.drop(columns=['speech'])
    print('Data fetched successfully!')

    df.to_csv('df.csv')
    t2 = perf_counter()

    print(f"It took {t2-t1} seconds")

    return df


try:
  df = pd.read_csv('df.csv')
except FileNotFoundError:
  df = fetch_data_from_kaggle()

In [ ]:
df['label'].value_counts()

In [ ]:
sns.countplot(df, x='label')

In [ ]:
# Select first 'fear' emotion audio sample
emotion = 'fear'
audio_data = np.array(df[df['label']==emotion]['data'])[0]      # Get waveform
sample_rate = np.array(df[df['label']==emotion]['sampling_rate'])[0]  # Get sample rate

# Visualize
waveplot(audio_data, sample_rate, emotion)    # Plot waveform
spectrogram(audio_data, sample_rate, emotion) # Plot spectrogram

In [ ]:
# same for the next few cells

emotion = 'fear'
a = np.array(df[df['label']==emotion]['data'])[0]
b = np.array(df[df['label']==emotion]['sampling_rate'])[0]


waveplot(a, b, emotion)
spectrogram(a, b, emotion)

In [ ]:
emotion = 'angry'
a = np.array(df[df['label']==emotion]['data'])[0]
b = np.array(df[df['label']==emotion]['sampling_rate'])[0]


waveplot(a, b, emotion)
spectrogram(a, b, emotion)

In [ ]:
emotion = 'disgust'
a = np.array(df[df['label']==emotion]['data'])[0]
b = np.array(df[df['label']==emotion]['sampling_rate'])[0]


waveplot(a, b, emotion)
spectrogram(a, b, emotion)

In [ ]:
emotion = 'neutral'
a = np.array(df[df['label']==emotion]['data'])[0]
b = np.array(df[df['label']==emotion]['sampling_rate'])[0]


waveplot(a, b, emotion)
spectrogram(a, b, emotion)

In [ ]:
emotion = 'sad'
a = np.array(df[df['label']==emotion]['data'])[0]
b = np.array(df[df['label']==emotion]['sampling_rate'])[0]


waveplot(a, b, emotion)
spectrogram(a, b, emotion)

In [ ]:
emotion = 'ps'
a = np.array(df[df['label']==emotion]['data'])[0]
b = np.array(df[df['label']==emotion]['sampling_rate'])[0]


waveplot(a, b, emotion)
spectrogram(a, b, emotion)

In [ ]:
emotion = 'happy'
a = np.array(df[df['label']==emotion]['data'])[0]
b = np.array(df[df['label']==emotion]['sampling_rate'])[0]


waveplot(a, b, emotion)
spectrogram(a, b, emotion)

In [ ]:
import numpy as np
import librosa

def extract_mfcc(df: pd.DataFrame) -> np.ndarray:
    """Extract MFCC features from audio data in DataFrame.

    Returns:
        np.ndarray: Array of MFCC features with shape (n_samples, n_mfcc)
    """
    y = np.array(df['data'])
    sr = np.array(df['sampling_rate'])

    all_mfccs = []

    for audio_samples, sample_rate in zip(y, sr):
        audio_samples = audio_samples.astype(np.float32)
        audio_samples /= np.max(np.abs(audio_samples))

        mfcc = librosa.feature.mfcc(
            y=audio_samples,
            sr=sample_rate,
            n_mfcc=40,
            n_fft=2048,
            hop_length=512
        )
        all_mfccs.append(np.mean(mfcc.T, axis=0))

    return np.array(all_mfccs)

# Usage
mfcc_features = extract_mfcc(df)
print(f"Extracted {len(mfcc_features)} MFCC feature vectors")

In [ ]:
mfcc_features

In [ ]:
# Convert list of MFCC features to numpy array
X = [x for x in mfcc_features]  # Creates a list copy (optional step)
X = np.array(X)  # Convert to numpy array

# Check array dimensions
print(X.shape)  # Output: (n_samples, n_mfcc_coefficients)

In [ ]:
# input split
X = np.expand_dims(X, -1)
X.shape

In [ ]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder()
y = enc.fit_transform(df[['label']])   # transforming categorical values

In [ ]:
y = y.toarray()   # sparse_output=False

In [ ]:
y.shape

In [ ]:
### LSTM Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt


# Define the LSTM model
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    """LSTM model for sequence classification"""
    def __init__(self, input_size=1, hidden_size=256, num_layers=1, num_classes=7):
        super().__init__()
        # LSTM layer - processes sequential data
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        # Fully connected layers with dropout for regularization
        self.dropout1 = nn.Dropout(0.2)  # 20% neuron dropout
        self.fc1 = nn.Linear(hidden_size, 128)

        self.dropout2 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 64)

        self.dropout3 = nn.Dropout(0.2)
        self.fc3 = nn.Linear(64, num_classes)  # Final output layer

        # Activation functions
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)  # Converts to probabilities

    def forward(self, x):
        # Input shape: (batch_size, seq_length, input_size)

        # LSTM processing - returns all hidden states
        lstm_out, _ = self.lstm(x)

        # Take only the last timestep's output
        lstm_out = lstm_out[:, -1, :]  # Shape: (batch_size, hidden_size)

        # Feed through FC layers
        x = self.dropout1(lstm_out)
        x = self.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.relu(self.fc2(x))
        x = self.dropout3(x)
        x = self.fc3(x)

        return self.softmax(x)  # Output probabilities

In [ ]:
X_tensor = torch.FloatTensor(X)  # Shape: (batch_size, seq_length=40, input_size=1)
y_tensor = torch.LongTensor(np.argmax(y, axis=1))  # Convert one-hot to class indices

# Create dataset and split for validation
dataset = TensorDataset(X_tensor, y_tensor)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# Initialize model
ser_model = LSTMModel(input_size=1, hidden_size=256, num_classes=7)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(ser_model.parameters())

In [ ]:
# Initialize lists to store metrics

train_losses = []
train_accs = []
val_losses = []
val_accs = []

num_epochs = 30

# Modified training loop to store metrics
for epoch in range(num_epochs):
  # Training
  ser_model.train()
  train_loss = 0.0
  train_correct = 0
  for inputs, labels in train_loader:
    optimizer.zero_grad()
    outputs = ser_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    _, predicted = torch.max(outputs.data, 1)
    train_correct += (predicted == labels).sum().item()

    # Validation
    ser_model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
      for inputs, labels in val_loader:
        outputs = ser_model(inputs)
        loss = criterion(outputs, labels)
        val_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        val_correct += (predicted == labels).sum().item()

    # Calculate metrics
  train_epoch_loss = train_loss/len(train_loader)
  train_epoch_acc = train_correct/train_size
  val_epoch_loss = val_loss/len(val_loader)
  val_epoch_acc = val_correct/val_size

    # Store metrics
  train_losses.append(train_epoch_loss)
  train_accs.append(train_epoch_acc)
  val_losses.append(val_epoch_loss)
  val_accs.append(val_epoch_acc)

  print(f'Epoch {epoch+1}/{num_epochs}:')
  print(f'Train Loss: {train_epoch_loss:.4f}, Acc: {train_epoch_acc:.4f}')
  print(f'Val Loss: {val_epoch_loss:.4f}, Acc: {val_epoch_acc:.4f}\n')

In [ ]:
# inference

# inference_model = LSTMModel()

# # Load the saved state_dict
# PATH = "ser_model_parameters.pth"
# inference_model.load_state_dict(torch.load(PATH, weights_only=True))

# # Set the model to evaluation mode for inference
# inference_model.eval()

In [ ]:
# You can pass this cell, I used it for training.

file_path = 'ser_model_parameters.pth'

# Save the model's state_dict
torch.save(ser_model.state_dict(), file_path)

In [ ]:
Audio('/content/orxan_ses.wav')

In [ ]:
Audio('/content/finally.mp3')

In [ ]:
# Plotting
plt.figure(figsize=(12, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Accuracy')
plt.plot(val_accs, label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Get all predictions
all_preds = []
all_labels = []
with torch.no_grad():
  for inputs, labels in val_loader:
    outputs = ser_model(inputs)
    _, preds = torch.max(outputs, 1)
    all_preds.extend(preds.numpy())
    all_labels.extend(labels.numpy())

# Plot confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds))

In [ ]:
!zip -r vad_ser_project.zip ser_model_parameters.pth df.csv orxan_ses.wav finally.mp3